# Laboratory 3 - Transforming Textual and Image Data into a Machine-Understandable Format

**Course:** Introduction to Artificial Intelligence (AI-101L) · **Instructor:** Dr. Muhammad Awais

---

## Learning Objectives

On completion of this laboratory you will be able to:

1. Explain why raw text and raw images cannot be fed directly to a traditional machine learning algorithm.
2. Apply the **Bag-of-Words** model with `CountVectorizer` to convert a corpus into a numerical document-term matrix.
3. Convert an image into a numerical array of pixel values with **Pillow** and **NumPy**.
4. Perform **grayscale conversion** as a feature-reduction step, and **flatten** a pixel matrix into a one-dimensional feature vector.
5. State precisely **what information each transformation discards**.

## Background

Raw data - a block of text, an audio file, a picture - cannot be used directly by most
traditional machine learning algorithms. Those algorithms require a **numerical, vector-based
input**. *Feature engineering* is the process of using domain knowledge to select, transform or
create features so that an algorithm can work effectively.

Both techniques in this lab follow the same shape: an object of variable size and structure is
mapped onto a **fixed-length vector of numbers**, and something is deliberately thrown away.

```
Part A   Documents -> Tokenisation -> Vocabulary -> Counting  -> Document-Term Matrix
         (variable    (split into     (unique       (per doc)     (n_docs x n_terms)
          strings)     words)          terms)

Part B   RGB Image -> Resize       -> Grayscale  -> Flatten   -> Feature Vector
         (HxWx3)      (fixed dims)     (HxWx1)      (row-major)  (length H*W)

                 both paths end in a fixed-length numeric vector
```

> defined in the notebook, and the image is either a student's own file or a generated dummy
> image. The lab therefore runs on any machine, offline.

### Setup and prerequisites

| Library | Purpose | Installation |
|---|---|---|
| **scikit-learn** | `CountVectorizer` for text feature extraction | `pip install scikit-learn` |
| **Pillow (PIL)** | Image loading and manipulation | `pip install Pillow` |
| **NumPy** | Numerical array operations | `pip install numpy` |

In [1]:
!pip install scikit-learn Pillow numpy pandas matplotlib

---
# Part A - Feature engineering for text data

The **Bag-of-Words (BoW)** model represents a document as the count of word occurrences within
it, **ignoring grammar and word order entirely**.

## Task 3.1 - Vectorising text with `CountVectorizer`

**Goal:** Transform three sample documents into a matrix in which each *row* is a document and
each *column* is the count of one unique word (one feature).

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# 1. Define Sample Text Data (our "corpus")
documents = [
    "The sun is shining today.",
    "The weather is good, the sun is great.",
    "A sunny day is a wonderful day."
]

print("--- Raw Text Documents ---")
for i, doc in enumerate(documents):
    print(f"Doc {i+1}: {doc}")

--- Raw Text Documents ---
Doc 1: The sun is shining today.
Doc 2: The weather is good, the sun is great.
Doc 3: A sunny day is a wonderful day.


In [3]:
# 2. Apply Feature Engineering: CountVectorizer (Bag-of-Words)
#    CountVectorizer performs tokenization AND vocabulary building in one step.
vectorizer = CountVectorizer()

# fit_transform learns the vocabulary and converts the documents into feature vectors
X_text = vectorizer.fit_transform(documents)

# 3. Analyze the Results
feature_names = vectorizer.get_feature_names_out()
text_matrix = X_text.toarray()

print("--- Bag-of-Words (BoW) Transformation ---")
print(f"Vocabulary (Features): {feature_names}")
print(f"Shape of Feature Matrix: {text_matrix.shape}   (documents x unique terms)")

--- Bag-of-Words (BoW) Transformation ---
Vocabulary (Features): ['day' 'good' 'great' 'is' 'shining' 'sun' 'sunny' 'the' 'today' 'weather'
 'wonderful']
Shape of Feature Matrix: (3, 11)   (documents x unique terms)


In [4]:
# A DataFrame view, purely so the matrix is readable
df_text = pd.DataFrame(text_matrix, columns=feature_names,
                       index=[f"Doc {i+1}" for i in range(len(documents))])

print("BoW Numerical Feature Matrix (Machine Understandable Format):\n")
display(df_text)

BoW Numerical Feature Matrix (Machine Understandable Format):


,day,good,great,is,shining,sun,sunny,the,today,weather,wonderful
Doc 1,0,0,0,1,1,1,0,1,1,0,0
Doc 2,0,1,1,2,0,1,0,2,0,1,0
Doc 3,2,0,0,1,0,0,1,0,0,0,1


In [5]:
# Prove the two silent decisions to yourself rather than taking them on trust.

print("Is 'a' in the vocabulary? ", 'a' in feature_names)
print("Is 'the' in the vocabulary?", 'the' in feature_names)
print()
print("Default token pattern :", vectorizer.token_pattern)
print("Lowercase enabled     :", vectorizer.lowercase)
print()
print("The token pattern requires 2+ word characters, which is why 'a' is dropped.")

Is 'a' in the vocabulary?  False
Is 'the' in the vocabulary? True

Default token pattern : (?u)\b\w\w+\b
Lowercase enabled     : True

The token pattern requires 2+ word characters, which is why 'a' is dropped.


In [6]:
# Sparsity: how much of this matrix is zero? This is the defining property of BoW.
total_cells = text_matrix.size
zero_cells = (text_matrix == 0).sum()

print(f"Matrix cells : {total_cells}")
print(f"Zero cells   : {zero_cells}")
print(f"Sparsity     : {zero_cells / total_cells:.1%}")
print()
print("With a real corpus of 20,000 documents this figure exceeds 99%,")
print("which is why scikit-learn returns a SPARSE matrix, not a dense array.")
print("Type returned by fit_transform:", type(X_text))

Matrix cells : 33
Zero cells   : 18
Sparsity     : 54.5%

With a real corpus of 20,000 documents this figure exceeds 99%,
which is why scikit-learn returns a SPARSE matrix, not a dense array.
Type returned by fit_transform: <class 'scipy.sparse._csr.csr_matrix'>


## Discussion questions (Part A)

Answer these in the Markdown cell below.

1. What does each **column** in the resulting matrix represent?
2. In this specific numerical matrix, what is the *machine-understandable format*?
3. How is this format an **oversimplification** of the original text data? (Consider what information has been lost.)

**Your answers (Part A):**

1. **What each column represents:** Each column in the matrix represents one unique vocabulary word extracted from the corpus (sorted alphabetically from `'day'` to `'wonderful'`). The numbers inside a column show how many times that specific word appeared in each document (`Doc 1`, `Doc 2`, and `Doc 3`). Note that single-letter words like `'a'` are ignored by default because `CountVectorizer` only keeps tokens with at least 2 characters.

2. **What the 'machine-understandable format' is:** The machine-understandable format here is a 2D numerical array (shape `3 x 11`), where each document is converted into a fixed-length row vector of 11 integer word counts (for example, `Doc 1` becomes `[0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0]`). Since ML algorithms only do math on numbers and cannot read strings directly, this integer vector format allows us to pass text into standard classifiers.

3. **Why this is an oversimplification (lost information):** Bag-of-Words throws away **word order, grammar, and punctuation**, so sentences with completely opposite meanings like *"The weather is good, not bad"* and *"The weather is bad, not good"* get the exact same vector. It also loses **semantic similarity** (it treats `'sun'` and `'sunny'`, or `'good'` and `'great'`, as totally unrelated columns), drops capitalization (`'The'` vs `'the'`), and ignores single-character words like `'a'` or `'I'`.

---

---
# In-Lab Exercises

**Exercise 1.** Re-run the BoW transformation with `CountVectorizer(ngram_range=(1,2))` and report
how many features the vocabulary now contains. Explain the growth.

**Exercise 2.** Add a fourth document that repeats the word *sun* five times. Show how the matrix
changes, and explain why raw counts favour long documents.

**Exercise 3.** Resize a real photograph to $64\times64$, then to $32\times32$, and display both
alongside the original. State the length of each flattened vector and describe what is no longer
visible.

**Exercise 4.** Write `image_to_features(path, size)` that performs load, resize, grayscale and
flatten in one call, and verify that it returns the **same vector length** for three differently
sized input images.

In [11]:
# --- Exercise 1: unigrams + bigrams --------------------------------------

vec_1gram = CountVectorizer(ngram_range=(1, 1))
vec_2gram = CountVectorizer(ngram_range=(1, 2))

X1 = vec_1gram.fit_transform(documents)
X2 = vec_2gram.fit_transform(documents)

print(f"ngram_range=(1,1) -> {X1.shape[1]:3d} features")
print(f"ngram_range=(1,2) -> {X2.shape[1]:3d} features")
print(f"Growth factor     : {X2.shape[1] / X1.shape[1]:.1f}x")
print()
print("New bigram features introduced:")
print([f for f in vec_2gram.get_feature_names_out() if ' ' in f])

ngram_range=(1,1) ->  11 features
ngram_range=(1,2) ->  24 features
Growth factor     : 2.2x

New bigram features introduced:
[np.str_('day is'), np.str_('good the'), np.str_('is good'), np.str_('is great'), np.str_('is shining'), np.str_('is wonderful'), np.str_('shining today'), np.str_('sun is'), np.str_('sunny day'), np.str_('the sun'), np.str_('the weather'), np.str_('weather is'), np.str_('wonderful day')]


In [12]:
# --- Exercise 2: raw counts favour long documents ------------------------

documents_4 = documents + ["sun sun sun sun sun and more sun today"]

vec = CountVectorizer()
M = vec.fit_transform(documents_4).toarray()
df4 = pd.DataFrame(M, columns=vec.get_feature_names_out(),
                   index=[f"Doc {i+1}" for i in range(len(documents_4))])
display(df4)

print("Total token count per document (row sums):")
print(df4.sum(axis=1).to_string())
print()
print("Doc 4 dominates the 'sun' column purely by repetition.")
print("This is exactly the problem TF-IDF (Lab 02) was designed to correct.")

,and,day,good,great,is,more,shining,sun,sunny,the,today,weather,wonderful
Doc 1,0,0,0,0,1,0,1,1,0,1,1,0,0
Doc 2,0,0,1,1,2,0,0,1,0,2,0,1,0
Doc 3,0,2,0,0,1,0,0,0,1,0,0,0,1
Doc 4,1,0,0,0,0,1,0,6,0,0,1,0,0


Total token count per document (row sums):
Doc 1    5
Doc 2    8
Doc 3    5
Doc 4    9

Doc 4 dominates the 'sun' column purely by repetition.
This is exactly the problem TF-IDF (Lab 02) was designed to correct.
